### Business Question
“Which supply chain factors—across products, regions, and shipping modes—are driving late deliveries, and what predictive signals can be used to reduce delays?”

## Graph Schema

### Nodes:
- Customer
- Order
- Product
- Department
- Region

### Relationships:
- Customer → PLACED → Order
- Order → CONTAINS → Product
- Product → IN_DEPARTMENT → Department
- Order → SHIPPED_TO → Region

The Order node contains key properties such as delay, payment type,
shipping mode, and sales.

## Note

The following Cypher queries were executed in Neo4j Browser.
They are included here for reproducibility of the pipeline.

In [2]:
from neo4j import GraphDatabase

uri = "bolt://localhost:7687"
username = "neo4j"
password = "Arya@189"

driver = GraphDatabase.driver(uri, auth=(username, password))

def run_query(query):
    with driver.session() as session:
        result = session.run(query)
        return [record.data() for record in result]

In [3]:
run_query("""
CREATE INDEX customer_id IF NOT EXISTS FOR (c:Customer) ON (c.id)
""")

run_query("""
CREATE INDEX order_id IF NOT EXISTS FOR (o:Order) ON (o.id)
""")

run_query("""
CREATE INDEX product_name IF NOT EXISTS FOR (p:Product) ON (p.name)
""")

run_query("""
CREATE INDEX department_name IF NOT EXISTS FOR (d:Department) ON (d.name)
""")

run_query("""
CREATE INDEX region_name IF NOT EXISTS FOR (r:Region) ON (r.name)
""")

[]

In [6]:
#Create Nodes
#Customers
run_query("""
LOAD CSV WITH HEADERS FROM 'file:///customers.csv' AS row
MERGE (c:Customer {id: row.customer_id})
""")

[]

In [7]:
#Orders
run_query("""
LOAD CSV WITH HEADERS FROM 'file:///orders.csv' AS row
MERGE (o:Order {id: row.order_id})
SET o.delay = toFloat(row.delay),
    o.shipping_mode = row.`Shipping Mode`,
    o.order_status = row.`Order Status`,
    o.payment_type = row.`Payment Method`,
    o.sales = toFloat(row.sales),
    o.quantity = toInteger(row.quantity)
""")

[]

In [8]:
#Products
run_query("""
LOAD CSV WITH HEADERS FROM 'file:///products.csv' AS row
MERGE (p:Product {name: row.product_name})
SET p.price = toFloat(row.product_price),
    p.image = row.product_image
""")

[]

In [9]:
#Departments
run_query("""
LOAD CSV WITH HEADERS FROM 'file:///product_department.csv' AS row
MERGE (d:Department {name: row.department})
""")

[]

In [10]:
#Region
run_query("""
LOAD CSV WITH HEADERS FROM 'file:///order_region.csv' AS row
WITH row
WHERE row.`Order Region` IS NOT NULL AND row.`Order Region` <> ""
MERGE (r:Region {name: row.`Order Region`})
""")

[]

In [11]:
#Create Relationships
#Customer -> Order

run_query("""
LOAD CSV WITH HEADERS FROM 'file:///customer_order.csv' AS row
MATCH (c:Customer {id: row.customer_id})
MATCH (o:Order {id: row.order_id})
MERGE (c)-[:PLACED]->(o)
""")

[]

In [12]:
#Order -> Product
run_query("""
LOAD CSV WITH HEADERS FROM 'file:///order_product.csv' AS row
MATCH (o:Order {id: row.order_id})
MATCH (p:Product {name: row.product_name})
MERGE (o)-[:CONTAINS]->(p)
""")

[]

In [13]:
#Product -> Department
run_query("""
LOAD CSV WITH HEADERS FROM 'file:///product_department.csv' AS row
MATCH (p:Product {name: row.product_name})
MATCH (d:Department {name: row.department})
MERGE (p)-[:IN_DEPARTMENT]->(d)
""")

[]

In [14]:
#Order -> Region
run_query("""
LOAD CSV WITH HEADERS FROM 'file:///order_region.csv' AS row
MATCH (o:Order {id: row.order_id})
MATCH (r:Region {name: row.`Order Region`})
MERGE (o)-[:SHIPPED_TO]->(r)
""")

[]

In [15]:
run_query("""
MATCH (n)
RETURN labels(n), count(n)
""")

[{'labels(n)': ['Order'], 'count(n)': 65752},
 {'labels(n)': ['Customer'], 'count(n)': 20652},
 {'labels(n)': ['Product'], 'count(n)': 118},
 {'labels(n)': ['Department'], 'count(n)': 11},
 {'labels(n)': ['Region'], 'count(n)': 23}]

In [16]:
run_query("""
MATCH ()-[r]->()
RETURN type(r), count(r)
""")

[{'type(r)': 'PLACED', 'count(r)': 65752},
 {'type(r)': 'CONTAINS', 'count(r)': 159763},
 {'type(r)': 'IN_DEPARTMENT', 'count(r)': 118},
 {'type(r)': 'SHIPPED_TO', 'count(r)': 65752}]

In [28]:
run_query("""
MATCH (a)-[r]->(b)
RETURN a, r, b LIMIT 10
""")

[{'a': {'id': '20755'},
  'r': ({'id': '20755'},
   'PLACED',
   {'order_status': 'COMPLETE',
    'delay': -1.0,
    'shipping_mode': 'Standard Class',
    'id': '77202'}),
  'b': {'order_status': 'COMPLETE',
   'delay': -1.0,
   'shipping_mode': 'Standard Class',
   'id': '77202'}},
 {'a': {'id': '19492'},
  'r': ({'id': '19492'},
   'PLACED',
   {'order_status': 'PENDING',
    'delay': 1.0,
    'shipping_mode': 'Standard Class',
    'id': '75939'}),
  'b': {'order_status': 'PENDING',
   'delay': 1.0,
   'shipping_mode': 'Standard Class',
   'id': '75939'}},
 {'a': {'id': '19491'},
  'r': ({'id': '19491'},
   'PLACED',
   {'order_status': 'CLOSED',
    'delay': 0.0,
    'shipping_mode': 'Standard Class',
    'id': '75938'}),
  'b': {'order_status': 'CLOSED',
   'delay': 0.0,
   'shipping_mode': 'Standard Class',
   'id': '75938'}},
 {'a': {'id': '19490'},
  'r': ({'id': '19490'},
   'PLACED',
   {'order_status': 'COMPLETE',
    'delay': -1.0,
    'shipping_mode': 'Standard Class',
   

## Graph Exploration

To understand the structure of our knowledge graph, we visualize a 
tree-like subgraph starting from a department node.

This helps us see how customers, orders, and products are connected.

### Cypher Query

```cypher
MATCH path =
(d:Department {name: "Fitness"})
<-[:IN_DEPARTMENT]-(p:Product)
<-[:CONTAINS]-(o:Order)
<-[:PLACED]-(c:Customer),
(o)-[:SHIPPED_TO]->(r:Region)
RETURN path LIMIT 10;

### Graph Output

![Graph](../images/graph_fitness.png)

### Graph Visualization Interpretation

The graph shows a subset of the supply chain centered around a product, illustrating how customers, orders, and departments are connected.

Customers place orders, which contain products, and each product belongs to a department. Order nodes include delay values, representing delivery performance.

This structure helps visualize how customer activity and product demand are linked to delivery delays, providing insight into key factors affecting supply chain performance.

In [21]:
#Region -> Delay
result = run_query("""
MATCH (o:Order)-[:SHIPPED_TO]->(r:Region)
RETURN r.name AS region, avg(o.delay) AS avg_delay
ORDER BY avg_delay DESC
""")

In [22]:
import pandas as pd

df = pd.DataFrame(result)
df

,region,avg_delay
0,Central Asia,0.646739
1,Central Africa,0.640288
2,South of USA,0.599257
3,South Asia,0.595202
4,Western Europe,0.590809
5,Eastern Europe,0.585913
6,East Africa,0.585644
7,West Asia,0.581602
8,East of USA,0.581145
9,Southeast Asia,0.576217


### Region vs Delay Analysis

This analysis shows that delivery delays vary across regions, with areas such as Central Asia and Central Africa experiencing the highest average delays, while regions like Canada and Southern Africa show relatively lower delays.

This suggests that geographic and logistical factors—such as infrastructure, distance, and regional supply chain complexity—play a significant role in delivery performance.

From a business perspective, region emerges as a key driver of delays and can be used as an important predictive feature to identify high-risk orders and optimize supply chain operations.

In [23]:
# department -> delay
run_query("""
MATCH (o:Order)-[:CONTAINS]->(p:Product)
      -[:IN_DEPARTMENT]->(d:Department)
RETURN d.name AS department, avg(o.delay) AS avg_delay
ORDER BY avg_delay DESC
""")

[{'department': 'Pet Shop', 'avg_delay': 0.7093495934959353},
 {'department': 'Technology', 'avg_delay': 0.5856655290102386},
 {'department': 'Fitness', 'avg_delay': 0.5808674503445481},
 {'department': 'Outdoors', 'avg_delay': 0.5731896819641581},
 {'department': 'Apparel', 'avg_delay': 0.5716806032191049},
 {'department': 'Footwear', 'avg_delay': 0.5680482178733535},
 {'department': 'Fan Shop', 'avg_delay': 0.5645114673829834},
 {'department': 'Golf', 'avg_delay': 0.5622927846327439},
 {'department': 'Discs Shop', 'avg_delay': 0.5493583415597236},
 {'department': 'Book Shop', 'avg_delay': 0.5333333333333333},
 {'department': 'Health and Beauty ', 'avg_delay': 0.5331491712707179}]

### Department vs Delay Analysis

Delivery delays vary by department, with Pet Shop showing the highest delays and categories like Health and Beauty having lower delays. 

This indicates that product category influences supply chain performance and can be used as a predictive factor for delays.

In [24]:
#shipping mode -> delay
run_query("""
MATCH (o:Order)
RETURN o.shipping_mode, avg(o.delay) AS avg_delay
ORDER BY avg_delay DESC
""")

[{'o.shipping_mode': 'Second Class', 'avg_delay': 2.0015651901706137},
 {'o.shipping_mode': 'First Class', 'avg_delay': 1.0},
 {'o.shipping_mode': 'Same Day', 'avg_delay': 0.48361803416409876},
 {'o.shipping_mode': 'Standard Class', 'avg_delay': -0.003102431085342261}]

### Shipping Mode vs Delay Analysis

The analysis shows that delivery delays vary significantly across shipping modes. Second Class shipments experience the highest delays, followed by First Class, while Same Day and Standard Class deliveries show much lower delays.

This indicates that slower shipping options are more prone to delays, possibly due to longer transit times, routing complexities, or lower prioritization in logistics operations.

From a business perspective, shipping mode is a critical factor influencing delivery performance and serves as a strong predictive feature for identifying and reducing potential delays in the supply chain.

In [25]:
run_query("""
MATCH (o:Order)
WHERE o.delay > 2
RETURN count(o) AS high_delay_orders
""")

[{'high_delay_orders': 5117}]

In [27]:
#product demand vs Delay
run_query("""
MATCH (o:Order)-[:CONTAINS]->(p:Product)
RETURN p.name, avg(o.delay) AS avg_delay, count(o) AS orders
ORDER BY avg_delay DESC LIMIT 10
""")

[{'p.name': 'sole e25 elliptical', 'avg_delay': 1.0, 'orders': 10},
 {'p.name': 'titleist club glove travel cover',
  'avg_delay': 0.9117647058823527,
  'orders': 34},
 {'p.name': "nike men's fingertrap max training shoe",
  'avg_delay': 0.9047619047619045,
  'orders': 63},
 {'p.name': 'garmin approach s4 golf gps watch',
  'avg_delay': 0.8529411764705882,
  'orders': 34},
 {'p.name': 'yakima doubledown ace hitch mount 4-bike rack',
  'avg_delay': 0.8125,
  'orders': 64},
 {'p.name': 'bowflex selecttech 1090 dumbbells',
  'avg_delay': 0.8,
  'orders': 10},
 {'p.name': 'ogio race golf shoes',
  'avg_delay': 0.7704918032786884,
  'orders': 61},
 {'p.name': 'titleist small wheeled travel cover',
  'avg_delay': 0.7592592592592595,
  'orders': 54},
 {'p.name': "adidas men's germany black crest away tee",
  'avg_delay': 0.7577854671280279,
  'orders': 289},
 {'p.name': 'taylormade white smoke in-12 putter',
  'avg_delay': 0.7460317460317456,
  'orders': 63}]

### Product vs Delay Analysis

The analysis shows that certain products consistently experience higher delivery delays, with items like the Sole E25 Elliptical and travel-related products showing the highest delays.

This suggests that product-specific factors such as size, handling requirements, or demand patterns can impact delivery performance.

From a business perspective, product-level insights help identify high-risk items and optimize inventory, shipping strategies, and fulfillment processes.